# Practical 1: The leaky integrate-and-fire neuron in NEST (`iaf_psc_alpha`)

In this notebook we treat the `iaf_psc_alpha` model as an "experimental preparation": we stimulate it with different protocols, record its membrane potential and spikes, and then wire neurons together to measure how strong a synapse is.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nest

print("NEST version:", nest.__version__)

## The model and its parameters

`iaf_psc_alpha` is a leaky integrate-and-fire neuron with alpha-shaped postsynaptic currents:

$$C_m \frac{dV}{dt} = -\frac{C_m}{\tau_m}(V - E_L) + I_{syn}(t) + I_e + I_{stim}(t)$$

When $V$ reaches $V_{th}$ a spike is emitted, $V$ is clamped to $V_{reset}$ for $t_{ref}$ ms. There is **no action potential waveform**: the spike is a formal event.

In [ ]:
nest.ResetKernel()
iaf_defaults = nest.GetDefaults("iaf_psc_alpha")

for key in ["C_m", "tau_m", "E_L", "V_th", "V_reset", "t_ref", "tau_syn_ex", "tau_syn_in", "I_e"]:
    print(f"{key:12s} = {iaf_defaults[key]}")
print("\nRecordables:", iaf_defaults["recordables"])

C_m = iaf_defaults["C_m"]        # pF
tau_m = iaf_defaults["tau_m"]    # ms
E_L = iaf_defaults["E_L"]        # mV
V_th = iaf_defaults["V_th"]      # mV
V_reset = iaf_defaults["V_reset"]
t_ref = iaf_defaults["t_ref"]    # ms
R_m = tau_m / C_m                # GOhm  (mV / pA)

**Rheobase.** With a constant current $I$, the membrane settles at $V_\infty = E_L + R_m I$ with $R_m = \tau_m / C_m$. The neuron fires only if $V_\infty > V_{th}$, so the minimal current (rheobase) is

$$I_{rheo} = \frac{C_m (V_{th} - E_L)}{\tau_m}$$

In [ ]:
I_rheo = C_m * (V_th - E_L) / tau_m
print(f"Input resistance R_m = {R_m*1000:.0f} MOhm")
print(f"Rheobase current     = {I_rheo:.1f} pA")

## DC current steps

We create one neuron per current amplitude and give each its own `dc_generator` (connected `one_to_one`). A single voltmeter and a single spike recorder record from all neurons at once.

Try to predict: what happens at exactly the rheobase? Just above it?

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

dc_amplitudes = [200.0, 350.0, 375.0, 380.0, 450.0, 800.0]   # pA
t_on, t_off, t_sim = 100.0, 600.0, 700.0                     # ms

neurons = nest.Create("iaf_psc_alpha", len(dc_amplitudes))
dc_gens = nest.Create("dc_generator", len(dc_amplitudes))
dc_gens.set([{"amplitude": a, "start": t_on, "stop": t_off} for a in dc_amplitudes])

voltmeter = nest.Create("voltmeter", params={"interval": 0.1})
spike_rec = nest.Create("spike_recorder")

nest.Connect(dc_gens, neurons, "one_to_one")
nest.Connect(voltmeter, neurons)
nest.Connect(neurons, spike_rec)

nest.Simulate(t_sim)

vm_ev = voltmeter.get("events")
sp_ev = spike_rec.get("events")

In [ ]:
fig, axes = plt.subplots(len(dc_amplitudes), 1, figsize=(9, 11), sharex=True)

for ax, gid, amp in zip(axes, neurons.tolist(), dc_amplitudes):
    mask = vm_ev["senders"] == gid
    spikes = sp_ev["times"][sp_ev["senders"] == gid]

    ax.plot(vm_ev["times"][mask], vm_ev["V_m"][mask], lw=1)
    ax.vlines(spikes, V_th, V_th + 15, color="r", lw=0.8)      # draw spikes as ticks
    ax.axhline(V_th, ls="--", c="grey", lw=0.8)
    ax.axvspan(t_on, t_off, color="orange", alpha=0.08)
    ax.set_ylim(E_L - 3, V_th + 17)
    ax.set_ylabel("V_m (mV)")
    ax.set_title(f"I = {amp:.0f} pA  ({amp/I_rheo:.2f} x rheobase) -> {len(spikes)} spikes", fontsize=9)

axes[-1].set_xlabel("time (ms)")
plt.tight_layout()
plt.show()

## A step-current protocol

A `step_current_generator` changes its amplitude at given times. We use it to apply, on one neuron:
a subthreshold step, a hyperpolarising step, two suprathreshold steps, and finally a very brief but strong pulse.

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

step_times  = [100.0, 300.0, 500.0, 700.0, 900.0, 1000.0, 1002.0]
step_values = [200.0, -200.0, 450.0, 800.0, 0.0, 5000.0, 0.0]      # pA
t_sim = 1100.0

neuron = nest.Create("iaf_psc_alpha")
step_gen = nest.Create("step_current_generator",
                       params={"amplitude_times": step_times, "amplitude_values": step_values})
voltmeter = nest.Create("voltmeter", params={"interval": 0.1})
spike_rec = nest.Create("spike_recorder")

nest.Connect(step_gen, neuron)
nest.Connect(voltmeter, neuron)
nest.Connect(neuron, spike_rec)
nest.Simulate(t_sim)

vm_ev = voltmeter.get("events")
spikes = spike_rec.get("events")["times"]

fig, (ax_v, ax_i) = plt.subplots(2, 1, figsize=(9, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax_v.plot(vm_ev["times"], vm_ev["V_m"], lw=1)
ax_v.vlines(spikes, V_th, V_th + 15, color="k", lw=0.8)
ax_v.axhline(V_th, ls="--", c="grey", lw=0.8)
ax_v.set_ylabel("V_m (mV)")

ax_i.step([0.0] + step_times + [t_sim], [0.0] + step_values + [0.0], where="post", c="tab:red")
ax_i.set_ylabel("I (pA)")
ax_i.set_xlabel("time (ms)")
ax_i.set_ylim(-400, 1000)            # the 5000 pA pulse is clipped for readability
plt.tight_layout()
plt.show()

## The f–I curve

We now sweep the current finely. Instead of generators, we use the neuron's own constant bias current `I_e`, set to a different value for each neuron in a single call. Spikes are counted only after an initial transient (spike recorder `start`).

For the LIF neuron the steady-state rate has a closed form:

$$f(I) = \left[\, t_{ref} + \tau_m \ln\frac{R_m I + E_L - V_{reset}}{R_m I + E_L - V_{th}} \right]^{-1} \quad \text{for } I > I_{rheo}$$

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

fi_amplitudes = np.arange(0.0, 1201.0, 20.0)     # pA
t_start_count, t_sim = 500.0, 2500.0
t_count = (t_sim - t_start_count) / 1000.0       # s

neurons = nest.Create("iaf_psc_alpha", len(fi_amplitudes))
neurons.set(I_e=fi_amplitudes.tolist())
spike_rec = nest.Create("spike_recorder", params={"start": t_start_count})
nest.Connect(neurons, spike_rec)
nest.Simulate(t_sim)

senders = spike_rec.get("events")["senders"]
rates_sim = np.array([np.sum(senders == gid) for gid in neurons.tolist()]) / t_count

rates_theory = np.zeros_like(fi_amplitudes)
supra = fi_amplitudes > I_rheo
RI = R_m * fi_amplitudes[supra]
rates_theory[supra] = 1000.0 / (t_ref + tau_m * np.log((RI + E_L - V_reset) / (RI + E_L - V_th)))

plt.figure(figsize=(6, 4))
plt.plot(fi_amplitudes, rates_theory, "k-", lw=1, label="analytical")
plt.plot(fi_amplitudes, rates_sim, "o", ms=4, label="NEST")
plt.axvline(I_rheo, ls="--", c="grey", lw=0.8, label="rheobase")
plt.xlabel("I_e (pA)")
plt.ylabel("firing rate (spikes/s)")
plt.legend()
plt.title("iaf_psc_alpha f–I curve")
plt.show()

## Noisy current injection

Real neurons receive fluctuating input. A `noise_generator` injects a current with a given mean and standard deviation (re-drawn every `dt` ms). We keep the **mean subthreshold** (300 pA < rheobase, i.e. $V_\infty = -58$ mV) and increase the noise amplitude.

We quantify the output with the firing rate and the **coefficient of variation** of the inter-spike intervals, $CV = \sigma_{ISI} / \mu_{ISI}$ ($CV=0$: clock-like; $CV \approx 1$: Poisson-like).

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

noise_std = [0.0, 50.0, 100.0, 200.0, 400.0]   # pA
I_mean = 300.0                                 # pA, below rheobase
t_sim = 10000.0

neurons = nest.Create("iaf_psc_alpha", len(noise_std))
noise_gens = nest.Create("noise_generator", len(noise_std))
noise_gens.set([{"mean": I_mean, "std": s, "dt": 1.0} for s in noise_std])

voltmeter = nest.Create("voltmeter", params={"interval": 0.1, "stop": 1000.0})   # store only the first second
spike_rec = nest.Create("spike_recorder")

nest.Connect(noise_gens, neurons, "one_to_one")
nest.Connect(voltmeter, neurons)
nest.Connect(neurons, spike_rec)
nest.Simulate(t_sim)

vm_ev = voltmeter.get("events")
sp_ev = spike_rec.get("events")

In [ ]:
fig, axes = plt.subplots(len(noise_std), 1, figsize=(9, 9), sharex=True)
rates, cvs = [], []

for ax, gid, s in zip(axes, neurons.tolist(), noise_std):
    mask = vm_ev["senders"] == gid
    spikes = np.sort(sp_ev["times"][sp_ev["senders"] == gid])
    isi = np.diff(spikes)
    rate = len(spikes) / (t_sim / 1000.0)
    cv = isi.std() / isi.mean() if len(isi) > 2 else np.nan
    rates.append(rate)
    cvs.append(cv)

    ax.plot(vm_ev["times"][mask], vm_ev["V_m"][mask], lw=0.7)
    ax.vlines(spikes[spikes < 1000], V_th, V_th + 15, color="k", lw=0.8)
    ax.axhline(V_th, ls="--", c="grey", lw=0.8)
    ax.set_ylim(E_L - 10, V_th + 17)
    ax.set_ylabel("V_m (mV)")
    ax.set_title(f"std = {s:.0f} pA:  rate = {rate:.1f} Hz,  CV = {cv:.2f}", fontsize=9)

axes[-1].set_xlabel("time (ms)")
plt.tight_layout()
plt.show()

## Synaptic bombardment with Poisson inputs

Instead of injecting current, we now stimulate the neuron **through synapses**: a `poisson_generator` represents a large population of independent presynaptic neurons. Each spike produces an alpha-shaped current whose peak equals the synaptic weight (in pA for `iaf_psc_alpha`).

The last neuron receives strong excitation *and* inhibition: its mean input is below rheobase but its fluctuations are large.

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

conditions = [
    {"label": "exc 4 kHz",               "r_ex": 4000.0,  "r_in": 0.0},
    {"label": "exc 7 kHz",               "r_ex": 7000.0,  "r_in": 0.0},
    {"label": "exc 10 kHz",              "r_ex": 10000.0, "r_in": 0.0},
    {"label": "exc 20 kHz + inh 3.5 kHz", "r_ex": 20000.0, "r_in": 3500.0},
]
w_ex, w_in = 10.0, -40.0     # pA
t_sim = 1000.0

neurons = nest.Create("iaf_psc_alpha", len(conditions))
pg_ex = nest.Create("poisson_generator", len(conditions))
pg_in = nest.Create("poisson_generator", len(conditions))
pg_ex.set([{"rate": c["r_ex"]} for c in conditions])
pg_in.set([{"rate": c["r_in"]} for c in conditions])

nest.Connect(pg_ex, neurons, "one_to_one", syn_spec={"weight": w_ex, "delay": 1.0})
nest.Connect(pg_in, neurons, "one_to_one", syn_spec={"weight": w_in, "delay": 1.0})

voltmeter = nest.Create("voltmeter", params={"interval": 0.1})
spike_rec = nest.Create("spike_recorder")
nest.Connect(voltmeter, neurons)
nest.Connect(neurons, spike_rec)
nest.Simulate(t_sim)

vm_ev = voltmeter.get("events")
sp_ev = spike_rec.get("events")

fig, axes = plt.subplots(len(conditions), 1, figsize=(9, 8), sharex=True)
for ax, gid, c in zip(axes, neurons.tolist(), conditions):
    mask = vm_ev["senders"] == gid
    spikes = sp_ev["times"][sp_ev["senders"] == gid]
    ax.plot(vm_ev["times"][mask], vm_ev["V_m"][mask], lw=0.7)
    ax.vlines(spikes, V_th, V_th + 15, color="k", lw=0.8)
    ax.axhline(V_th, ls="--", c="grey", lw=0.8)
    ax.set_ylim(E_L - 10, V_th + 17)
    ax.set_ylabel("V_m (mV)")
    ax.set_title(f"{c['label']}: {len(spikes)} spikes/s", fontsize=9)
axes[-1].set_xlabel("time (ms)")
plt.tight_layout()
plt.show()